# PoliMillionaire - Multi-BM25 retrieval (SimpleWiki + KELM), no LLM

Risolve i quiz basandosi soltatnto sulla ricerca in due diverse banche dati: SimpleWiki (1.6M di documenti lunghi) e KELM (500k asserzioni corte). Utilizza "Reciprocal Rank Fusion" per bilanciare l'importanza dei documenti estratti senza impazzire coi punteggi grezzi di BM25 che non sarebbero compatibili fra loro.

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [2]:
from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, run_all_competitions, summarize, write_logs

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

Logged in as NeuroniNegroni (student)


In [3]:
WIKI_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_bm25.joblib"
KELM_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "kelm_500k_bm25.joblib"

if not WIKI_INDEX_PATH.exists() or not KELM_INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing one of the indexes")

wiki_index = load_retrieval_index(WIKI_INDEX_PATH)
print("Loaded SimpleWiki index:", WIKI_INDEX_PATH)

kelm_index = load_retrieval_index(KELM_INDEX_PATH)
print("Loaded KELM index:", KELM_INDEX_PATH)

indexes = [wiki_index, kelm_index]

resource module not available on Windows


Loaded SimpleWiki index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\simplewiki_160w_bm25.joblib
Loaded KELM index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\kelm_500k_bm25.joblib


In [6]:
ATTEMPTS_PER_COMPETITION = 10
STRATEGY_NAME = "simplewiki_and_kelm_bm25_no_llm"

rows = run_all_competitions(
    client=client,
    index=indexes,  # Qui stiamo passando un array di 2 indici, la nostra funzione si occupera' di fare RRF
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_kelm_bm25_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

[simplewiki_and_kelm_bm25_no_llm] comp=0 Entertainment attempt=1 q=1 level=0 chosen=3 correct=False earned=0 latency=1.183997600004659
Q: What is the primary reason HAL 9000 malfunctions in '2001: A Space Odyssey'?
A: A malfunction in the spacecraft's computer system
Top evidence: Douglas Rain :: Douglas Rain (March 13, 1928 – November 11, 2018) was a Canadian actor and narrator.

He was known for providing the voice of the HAL 9000 computer in the movie 2001: A Space Odyssey and the sequel 2010.

He decided to study acting at the Banff School of Fine Arts in Banff, Alberta and the Old Vic School in Bristol, England. He was nominated for...

[simplewiki_and_kelm_bm25_no_llm] comp=0 Entertainment attempt=2 q=1 level=0 chosen=2 correct=False earned=0 latency=1.6562567000073614
Q: How does the sled in Citizen Kane relate to the film's central mystery 'Rosebud'?
A: It is a character in the film
Top evidence: Citizen Kane :: Citizen Kane is a 1941 American drama movie starring Orson Welles 

In [7]:
summarize(rows)


Summary
0 Entertainment: rows=12, sessions=10, correct=2, row_acc=0.167, max_seen_level=0, avg_latency=1.040s
1 Ancient History and Politics: rows=15, sessions=10, correct=5, row_acc=0.333, max_seen_level=0, avg_latency=1.210s
2 Science and Nature: rows=17, sessions=10, correct=7, row_acc=0.412, max_seen_level=0, avg_latency=1.092s
3 Maths: rows=17, sessions=10, correct=7, row_acc=0.412, max_seen_level=0, avg_latency=2.025s
